# Set 04 – Pipeline und ColumnTransformer

Die Vorverarbeitungsschritte werden zu einem wiederverwendbaren Ablauf verbunden. Es wird kein Klassifikationsmodell trainiert: Die Pipeline endet nach der Datenaufbereitung.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Gemischte Rohdaten

Numerische und kategoriale Spalten benötigen unterschiedliche Verarbeitungsschritte.

In [ ]:
roh_daten = pd.DataFrame({
    "alter": [21, 35, np.nan, 48, 57, 29, 42, np.nan],
    "monatsumsatz": [120, 260, 190, np.nan, 510, 175, 330, 285],
    "vertrag": ["Basis", "Plus", "Basis", "Premium", "Premium", np.nan, "Plus", "Basis"],
    "kanal": ["Web", "Filiale", "Web", "Partner", "Filiale", "Web", np.nan, "Partner"],
})
display(roh_daten)
print("Fehlende Werte:")
print(roh_daten.isna().sum())

## 2. Pipeline für numerische Spalten

Die Schritte laufen nacheinander: fehlende Zahlen durch den Median ersetzen und anschließend standardisieren.

In [ ]:
numerische_pipeline = Pipeline(steps=[
    ("fehlende_werte", SimpleImputer(strategy="median")),
    ("skalierung", StandardScaler()),
])

## 3. Pipeline für kategoriale Spalten

Fehlende Kategorien werden durch die häufigste Kategorie ersetzt und danach one-hot-codiert.

In [ ]:
kategoriale_pipeline = Pipeline(steps=[
    ("fehlende_werte", SimpleImputer(strategy="most_frequent")),
    ("kodierung", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

## 4. Spalten mit ColumnTransformer zuordnen

Der ColumnTransformer schickt jede Spaltengruppe durch die passende Pipeline und setzt die Ergebnisse zusammen.

In [ ]:
numerische_spalten = ["alter", "monatsumsatz"]
kategoriale_spalten = ["vertrag", "kanal"]

vorverarbeitung = ColumnTransformer(transformers=[
    ("numerisch", numerische_pipeline, numerische_spalten),
    ("kategorial", kategoriale_pipeline, kategoriale_spalten),
])

## 5. Gesamtablauf als Pipeline

In Set 05 kann später ein Modell als weiterer Schritt ergänzt werden, ohne die Vorverarbeitung umzubauen.

In [ ]:
pipeline = Pipeline(steps=[("vorverarbeitung", vorverarbeitung)])
transformierte_daten = pipeline.fit_transform(roh_daten)
spaltennamen = pipeline.named_steps["vorverarbeitung"].get_feature_names_out()
transformiert = pd.DataFrame(transformierte_daten, columns=spaltennamen)
display(transformiert.round(2))
print("Vorher:", roh_daten.shape)
print("Nachher:", transformiert.shape)

## 6. Gelernte Werte untersuchen

Über named_steps greifen wir auf innere Schritte zu.

In [ ]:
numerischer_imputer = (
    pipeline.named_steps["vorverarbeitung"]
    .named_transformers_["numerisch"]
    .named_steps["fehlende_werte"]
)
encoder = (
    pipeline.named_steps["vorverarbeitung"]
    .named_transformers_["kategorial"]
    .named_steps["kodierung"]
)
print("Gelernte Zahlen-Mediane:", numerischer_imputer.statistics_)
print("Gelernte Kategorien:", encoder.categories_)

## 7. Neue Rohdaten transformieren

Kiosk war in den Referenzdaten nicht enthalten. handle_unknown="ignore" erlaubt trotzdem eine Transformation mit identischer Ausgabeform.

In [ ]:
neue_daten = pd.DataFrame({
    "alter": [33, np.nan],
    "monatsumsatz": [240, 410],
    "vertrag": ["Plus", "Premium"],
    "kanal": ["Kiosk", "Web"],
})
neu_transformiert = pipeline.transform(neue_daten)
display(pd.DataFrame(neu_transformiert, columns=spaltennamen).round(2))

## Warum eine Pipeline?

- Die Reihenfolge ist dokumentiert.
- fit() und transform() werden konsistent verwendet.
- Neue Daten erhalten exakt dieselbe Verarbeitung.
- Später kann ein Modell ergänzt werden.
- Nach einer späteren Datenaufteilung wird nur auf Trainingsdaten gefittet und so Leakage vermieden.

Bis hierhin wurden ausschließlich Merkmale vorbereitet – ohne Zielvariable, Klassifikation oder Vorhersage.